# 03b — Connections between Košice and places abroad with one transfer

Journeys between the Košice hub and stations in Czechia, Hungary, Austria, Poland and Ukraine with **exactly one
change of trains**, both directions. Code: `traintranv2/nb03_foreign.py` on the engine of 02b
(`traintranv2/nb02_connections.py`).

| § | step | output (`output/kosice/foreign/`) |
|---|---|---|
| 1 | rules and runs | — |
| 2 | itineraries per country run; feeds, duplicates and transfers onto the same train | `one_transfer_itins.parquet`, `feed_quality.csv`, `cross_feed_duplicates.csv` |
| 3 | per station and per LAU, four counts (R1) and gateway dependence | `one_transfer_station.csv`, `one_transfer_lau.csv` |
| 4 | coverage: direct / newly reached / either, with denominators | `one_transfer_coverage.csv` |
| 5 | gateways: volume and exclusive catchment | `one_transfer_hubs.csv` |
| 6 | transfer waits and connections across midnight | — |
| 7 | sensitivity: transfer window, transfers between two stations of one city | `one_transfer_sensitivity.csv` |
| 8 | against the legacy run | — |

**Rules** (Reviewer 1, register CON-10, PIP-4 to PIP-10, PIP-15, PIP-17, STA-2, VAL-2):
* **The first leg is a ZSR train** between Košice and a transfer station T (in Slovakia or abroad). **The second
  leg** runs between T and a station of the destination country, on **ZSR_SK or the country's own feed(s)**: the
  Czech, Hungarian, Austrian, Polish (PKP Intercity + all other carriers) or Ukrainian timetable.
* **One run per destination country** on the window its feeds share with ZSR (table in 03a): CZ 364, HU 301, AT 276,
  UA 364 days; **PL 32 days** (5 June – 6 July 2026, both Polish feeds), plus **PL_IC alone on 101 days** as a
  sensitivity run (`PL:PL_IC`). Per-day figures divide by the run's own window.
* **A transfer** is at one station group (01b `station_group_id`): two feeds' records of one station are one place, so
  a change from a ZSR train to a Czech-feed train at Bohumín needs no extra matching. Wait between
  `min_connect_min` (10) and `max_connect_min` (90) minutes. No walk between two stations (PIP-17, measured in §7).
* **A train in two feeds counts once** (PIP-8): ZSR's copy of a Czech, Hungarian or Austrian train (same stations,
  departure and arrival within 2 minutes, a common date) is dropped and the national feed's leg kept, with the run
  dates of both.
* **No transfer onto the same train**: a "change" to the other feed's copy of the train one is on is dropped.
* **Trains published in pieces are joined**: BAHN_AT cuts its trains at region and border points (RJX Wien –
  Bratislava-Petržalka, then Petržalka – Bratislava hl.st.); pieces with one train number that meet at a station are
  one trip (`nb02_connections.join_train_pieces`), as legs joined by stay-seated transfers are (Polish feeds).
* **Across midnight** (PIP-6): a second train leaving after midnight on the next date connects to a first train
  arriving late on date d (`next_day`); no modulo-24 h wrap.
* Dated pairing, every destination kept, `has_direct`, the four counts and gateway dependence as in 02b.
* A station's LAU abroad comes from the national feed (03a).

In [1]:
import os, sys

# Run from the project root so every relative path resolves the same from notebooks/ or the root.
_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    os.chdir(os.path.dirname(_here))
sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

Working directory: E:\notes\api_zssk\project


In [2]:
import pandas as pd
import traintranv2  # noqa: F401  (PROJ data fix)
from traintranv2.config import default_config
from traintranv2 import nb02_connections as C
from traintranv2 import nb03_foreign as F
from traintranv2.nb01a_feeds import write_meta
from traintranv2.nb01b_stations import load_lau_subset

cfg = default_config()
OUT = os.path.join(cfg.out_dir, 'foreign')
os.makedirs(OUT, exist_ok=True)
tags = pd.read_csv(os.path.join(cfg.out_dir, 'station_tags.csv'), dtype=str)
L = C.lau_with_city_units(load_lau_subset(cfg), cfg)          # Košice / Bratislava districts -> one unit each
tags = C.tags_with_city_units(tags, cfg, L)
hub_lau = C.hub_city_laus(cfg)                                 # the whole city of Košice
snap = pd.read_csv(os.path.join(cfg.out_dir, 'station_rail_snap.csv'), dtype={'station_key': str})
geo = tags.set_index('station_key')
lau_of = F.station_lau(tags, cfg)
hubs = C.hub_groups(cfg, tags)
pop, pop_year = F.lau_population(L)
lau_geo = tags.drop_duplicates('lau_gisco_id').set_index('lau_gisco_id')[['lau_name_national', 'nuts3_id', 'degurba']]
runs = F.country_runs(cfg)
direct = pd.read_parquet(os.path.join(OUT, 'direct_legs.parquet'))     # 03a

## 1. Rules of this run

In [3]:
print(f'transfer window : {cfg.min_connect_min} .. {cfg.max_connect_min} min   (sensitivity grid: {cfg.sensitivity_grid})')
print(f'pairing rule    : {cfg.transfer_pairing};  city-transfer sensitivity: any two stations of one LAU, >= 45 min')
display(pd.DataFrame([{'run': r['run'], 'feeds': ' + '.join([cfg.domestic_feed] + r['feeds']), **F.run_window(r),
                       'headline': r['headline']} for r in runs]))

transfer window : 10 .. 90 min   (sensitivity grid: [(10, 60), (15, 60), (20, 60), (20, 90), (20, 120), (30, 120)])
pairing rule    : dated;  city-transfer sensitivity: any two stations of one LAU, >= 45 min


,run,feeds,window_start,window_end,window_days,headline
0,CZ,ZSR_SK + DRAHY_CZ,2025-12-14,2026-12-12,364,True
1,HU,ZSR_SK + MAV_HU,2026-01-19,2026-11-15,301,True
2,UA,ZSR_SK + UKRZ_UA,2025-12-14,2026-12-12,364,True
3,PL,ZSR_SK + PL_IC + PL_REG,2026-06-05,2026-07-06,32,True
4,PL:PL_IC,ZSR_SK + PL_IC,2026-05-18,2026-08-26,101,False
5,AT,ZSR_SK + BAHN_AT,2026-03-12,2026-12-12,276,True


## 2. Itineraries per country run

`feed_quality` (REP-4): first and second legs, ZSR copies of national trains removed, "transfers" onto the same train
removed, itineraries, those across midnight, transfer stations, and itineraries by the feed of the train abroad.

In [4]:
itins, dups, quality, TT = [], [], [], {}
for r in runs:
    tt = F.load_run_timetables(cfg, r, tags)
    TT[r['run']] = tt
    it, dup, stats = F.run_one_transfer(cfg, r, tt, tags, hubs)
    W = F.run_window(r)
    dl = direct[direct.run == r['run']]
    has = {d: set(dl.loc[dl.direction == d, 'station_key']) for d in ('inbound', 'outbound')}
    it['has_direct'] = [k in has[d] for k, d in zip(it['station_key'], it['direction'])]
    itins.append(it.assign(**W))
    dups.append(dup.assign(country=r['country'], **W))
    for d in ('inbound', 'outbound'):
        x = it[it.direction == d]
        abroad = x['leg2_feed'] if d == 'outbound' else x['leg1_feed']      # the train on the foreign side
        quality.append({'country': r['country'], 'run': r['run'], 'direction': d, **stats[d],
                        'same_train_dropped': stats['same_train_dropped'], 'itineraries': len(x),
                        'itineraries_next_day': int(x['next_day'].sum()), 'transfer_stations': x['T_key'].nunique(),
                        'second_leg_feeds': '; '.join(f'{f}: {n}' for f, n in abroad.value_counts().items()), **W})
itins = pd.concat(itins, ignore_index=True)
dups = pd.concat(dups, ignore_index=True)
quality = pd.DataFrame(quality)
display(quality.drop(columns=['window_start', 'window_end']))
gname = tags.drop_duplicates('station_group_id').set_index('station_group_id')['stop_name']
print('trains present in two feeds (copies dropped), by feed pair and the busiest station pairs:')
display(dups.groupby(['run', 'drop_feed', 'keep_feed']).size().rename('legs dropped').to_frame())
display(dups.assign(pair=dups['from_group'].map(gname) + ' -> ' + dups['to_group'].map(gname))
        .groupby(['run', 'pair']).size().sort_values(ascending=False).head(10).rename('legs').to_frame())

,country,run,direction,first_legs,second_legs,duplicates_dropped,same_train_dropped,itineraries,itineraries_next_day,transfer_stations,second_leg_feeds,window_days
0,CZ,CZ,inbound,2359,153116,2768,471,21284,368,46,DRAHY_CZ: 19392; ZSR_SK: 1892,364
1,CZ,CZ,outbound,2315,154244,2736,471,24566,7829,37,DRAHY_CZ: 22880; ZSR_SK: 1686,364
2,HU,HU,inbound,2359,41550,613,584,11884,346,15,MAV_HU: 11748; ZSR_SK: 136,301
3,HU,HU,outbound,2315,37712,613,584,7444,14,16,MAV_HU: 7282; ZSR_SK: 162,301
4,UA,UA,inbound,2359,146,4,4,17,0,15,UKRZ_UA: 17,364
5,UA,UA,outbound,2315,111,4,4,15,0,4,UKRZ_UA: 13; ZSR_SK: 2,364
6,PL,PL,inbound,2013,8858,4,0,348,66,10,PL_REG: 310; ZSR_SK: 38,32
7,PL,PL,outbound,1972,8029,4,0,1558,1053,11,PL_IC: 1023; PL_REG: 438; ZSR_SK: 97,32
8,PL,PL:PL_IC,inbound,2239,14176,0,0,53,0,2,ZSR_SK: 53,101
9,PL,PL:PL_IC,outbound,2198,14173,0,0,3402,3283,5,PL_IC: 3283; ZSR_SK: 119,101


trains present in two feeds (copies dropped), by feed pair and the busiest station pairs:


,,,legs dropped
run,drop_feed,keep_feed,
AT,ZSR_SK,BAHN_AT,264
CZ,ZSR_SK,DRAHY_CZ,5504
HU,ZSR_SK,MAV_HU,1226
PL,ZSR_SK,PL_REG,8
UA,ZSR_SK,UKRZ_UA,8


legs
run pair                                  
CZ  Praha hl.n. -> Pardubice hl.n.      64
    Kolín -> Pardubice hl.n.            54
    Pardubice hl.n. -> Praha hl.n.      54
    Pardubice hl.n. -> Kolín            50
    Praha hl.n. -> Kolín                48
    Pardubice hl.n. -> Olomouc hl.n.    42
    Praha hl.n. -> Olomouc hl.n.        42
    Kolín -> Praha hl.n.                40
    Ostrava-Svinov -> Ostrava hl.n.     40
    Olomouc hl.n. -> Pardubice hl.n.    34

## 3. Per station and per LAU

Four counts per unit (R1): distinct **journeys**, the distinct **first** and **second trains** behind them, and the
journeys **sharing a train** with another journey to the same unit. Gateway dependence (PIP-15): the main transfer
station's share of the LAU's journeys per day, and the fewest stations carrying 90 %.

In [5]:
station, lau, per_hub = [], [], []
for r in runs:
    N, W = len(r['days']), F.run_window(r)
    dl = direct[direct.run == r['run']]
    for d in ('inbound', 'outbound'):
        x = itins[(itins.run == r['run']) & (itins.direction == d)]
        if x.empty:
            continue
        s = C.summarise_one_transfer(x, 'station_key', N).assign(direction=d, country=r['country'], run=r['run'], **W)
        s['has_direct'] = s['station_key'].isin(set(dl.loc[dl.direction == d, 'station_key']))
        station.append(s)
        l = C.summarise_one_transfer(x, 'lau_gisco_id', N).assign(direction=d, country=r['country'], run=r['run'], **W)
        l['has_direct'] = l['lau_gisco_id'].isin(set(dl.loc[dl.direction == d, 'lau_gisco_id']))
        ph, hs = C.hub_dependence(x, 'lau_gisco_id', N, share=0.9)
        l = l.merge(hs[['lau_gisco_id', 'main_hub', 'main_hub_share', 'hubs_for_share']], on='lau_gisco_id', how='left')
        lau.append(l.rename(columns={'hubs_for_share': 'hubs_for_90'}))
        per_hub.append(ph.assign(direction=d, country=r['country'], run=r['run'], **W))
station = pd.concat(station, ignore_index=True)
station['lau_gisco_id'] = station['station_key'].map(lau_of)
station = (station.join(geo['stop_name'], on='station_key').join(lau_geo, on='lau_gisco_id')
           .join(snap.set_index('station_key')['rail_km'], on='station_key'))
station = station[['country', 'run', 'station_key', 'stop_name', 'lau_gisco_id', 'lau_name_national', 'nuts3_id', 'degurba',
                   'direction', 'n_journeys', 'journeys_per_day', 'service_days', 'service_share', 'min_time_min',
                   'median_time_min', 'n_first_legs', 'n_second_legs', 'n_shared_leg', 'n_hubs', 'median_wait_min',
                   'has_direct', 'rail_km', 'window_start', 'window_end', 'window_days']]
lau = pd.concat(lau, ignore_index=True).join(lau_geo, on='lau_gisco_id')
lau['population'] = lau['lau_gisco_id'].map(pop)
lau['pop_year'] = lau['lau_gisco_id'].map(pop_year)
lau = lau[['country', 'run', 'lau_gisco_id', 'lau_name_national', 'nuts3_id', 'degurba', 'population', 'pop_year',
           'direction', 'n_journeys', 'journeys_per_day', 'service_days', 'service_share', 'min_time_min', 'median_time_min',
           'n_first_legs', 'n_second_legs', 'n_shared_leg', 'n_hubs', 'median_wait_min', 'has_direct', 'main_hub',
           'main_hub_share', 'hubs_for_90', 'window_start', 'window_end', 'window_days']]
per_hub = pd.concat(per_hub, ignore_index=True)
print('largest LAUs reached with one transfer, inbound, per headline run:')
for r in runs:
    if r['headline']:
        x = lau[(lau.run == r['run']) & (lau.direction == 'inbound')].sort_values('population', ascending=False).head(5)
        display(x[['run', 'lau_name_national', 'population', 'journeys_per_day', 'median_time_min', 'median_wait_min',
                   'has_direct', 'main_hub', 'main_hub_share', 'hubs_for_90']])

largest LAUs reached with one transfer, inbound, per headline run:


,run,lau_name_national,population,journeys_per_day,median_time_min,median_wait_min,has_direct,main_hub,main_hub_share,hubs_for_90
226,CZ,Praha,1407084.0,86.538462,557.0,30.0,True,ZSR_SK:5457076,0.413,11
157,CZ,Brno,404296.0,32.217033,553.0,41.0,False,ZSR_SK:5613206,0.280,7
11,CZ,Ostrava,280853.0,88.005495,371.0,32.0,True,DRAHY_CZ:czptt:stop:CZ:34434,0.171,14
369,CZ,Plzeň,187863.0,7.324176,612.0,23.0,False,DRAHY_CZ:czptt:stop:CZ:57076,1.000,1
424,CZ,Liberec,108214.0,1.406593,667.0,49.5,False,DRAHY_CZ:czptt:stop:CZ:53613,0.998,1


,run,lau_name_national,population,journeys_per_day,median_time_min,median_wait_min,has_direct,main_hub,main_hub_share,hubs_for_90
1116,HU,Budapest,1675777.0,48.534884,276.0,30.0,True,MAV_HU:267,0.399,4
1198,HU,Debrecen,202331.0,6.182724,437.0,65.0,False,MAV_HU:928,0.553,2
1053,HU,Miskolc,139246.0,34.993355,234.0,42.0,True,MAV_HU:869,0.200,5
1174,HU,Győr,128225.0,4.318937,378.0,56.0,False,MAV_HU:267,1.000,1
1084,HU,Nyíregyháza,114096.0,14.767442,366.0,30.0,False,MAV_HU:945,0.324,4


,run,lau_name_national,population,journeys_per_day,median_time_min,median_wait_min,has_direct,main_hub,main_hub_share,hubs_for_90
1391,UA,Мукачівська,110037.0,1.074176,503.0,38.0,True,UKRZ_UA:5613860,0.839,2
1390,UA,Чопська,15410.0,6.920330,523.5,38.0,True,UKRZ_UA:5613860,0.130,11


,run,lau_name_national,population,journeys_per_day,median_time_min,median_wait_min,has_direct,main_hub,main_hub_share,hubs_for_90
1409,PL,Warszawa,1866729.0,2.03125,831.0,62.0,False,ZSR_SK:5613206,0.492,4
1401,PL,Kraków,816614.0,4.75000,841.0,57.0,False,PL_REG:129059,0.414,4
1411,PL,Katowice,277415.0,1.00000,737.0,68.0,False,ZSR_SK:5613206,1.000,1
1416,PL,Rzeszów,199436.0,1.93750,1015.5,50.0,False,PL_REG:129059,0.500,2
1412,PL,Sosnowiec,182971.0,1.00000,768.0,68.0,False,ZSR_SK:5613206,1.000,1


,run,lau_name_national,population,journeys_per_day,median_time_min,median_wait_min,has_direct,main_hub,main_hub_share,hubs_for_90
1505,AT,Wien,2040914.0,28.271739,454.0,64.0,False,BAHN_AT:Psk:56:1,0.789,2
1513,AT,Graz,306971.0,0.032609,569.0,41.0,False,BAHN_AT:hu:14216:27001535:27001419,1.000,1
1512,AT,Linz,214272.0,2.920290,544.0,56.0,False,BAHN_AT:hu:14216:27001535:27001419,0.651,2
1517,AT,Salzburg,157834.0,2.902174,624.0,56.0,False,BAHN_AT:hu:14216:27001535:27001419,0.655,2
1518,AT,Innsbruck,132820.0,1.949275,804.0,64.0,False,BAHN_AT:Psk:56:1,0.513,2


## 4. Coverage, with its denominators

Per country: **all** LAUs, LAUs **with a station** of the run's feeds, **direct**, **one transfer**, **newly reached**
(one transfer, no direct train), **direct or one transfer**.

In [6]:
rows = []
for r in runs:
    c = r['country']
    allc = set(L.loc[L['CNTR_CODE'] == c, 'GISCO_ID'])
    feeds = [cfg.domestic_feed] + r['feeds']
    ws = {lau_of[k] for k in tags.loc[tags.feed_key.isin(feeds) & (tags.nuts_rg_01_country_id == c), 'station_key']}
    for d in ('inbound', 'outbound'):
        dl = set(direct.loc[(direct.run == r['run']) & (direct.direction == d), 'lau_gisco_id'])
        ol = set(lau.loc[(lau.run == r['run']) & (lau.direction == d), 'lau_gisco_id'])
        for g, ids in [('all_lau', allc), ('with_station', ws), ('direct', dl), ('one_transfer', ol),
                       ('newly_reached', ol - dl), ('direct_or_one_transfer', ol | dl)]:
            ids = ids & allc
            rows.append({'country': c, 'run': r['run'], 'direction': d, 'group': g, 'n_lau': len(ids),
                         'population': pop.reindex(list(ids)).sum(), **F.run_window(r)})
cov = pd.DataFrame(rows)
base = cov[cov.group == 'all_lau'].set_index(['run', 'direction'])
k = list(zip(cov['run'], cov['direction']))
cov['share_lau'] = (cov['n_lau'] / base['n_lau'].reindex(k).to_numpy()).round(4)
cov['share_population'] = (cov['population'] / base['population'].reindex(k).to_numpy()).round(5)
cov = cov[['country', 'run', 'direction', 'group', 'n_lau', 'population', 'share_lau', 'share_population',
           'window_start', 'window_end', 'window_days']]
display(cov[cov.direction == 'inbound'].pivot_table(index='run', columns='group', values='n_lau', aggfunc='sum'))
display(cov[cov.direction == 'inbound'].pivot_table(index='run', columns='group', values='population', aggfunc='sum')
        .map(lambda v: f'{v:,.0f}'))

group,all_lau,direct,direct_or_one_transfer,newly_reached,one_transfer,with_station
run,,,,,,
AT,2092,0,31,31,31,620
CZ,6258,14,535,521,535,1675
HU,3155,10,186,176,186,757
PL,2477,0,21,21,21,1187
PL:PL_IC,2477,0,11,11,11,416
UA,1769,3,3,0,2,53


group,all_lau,direct,direct_or_one_transfer,newly_reached,one_transfer,with_station
run,,,,,,
AT,"9,215,956",0,"3,160,473","3,160,473","3,160,473","6,245,008"
CZ,"10,915,839","2,119,181","6,068,419","3,949,238","6,068,419","8,189,189"
HU,"9,488,428","1,865,994","3,844,552","1,978,558","3,844,552","6,691,792"
PL,"37,332,510",0,"4,193,156","4,193,156","4,193,156","27,953,638"
PL:PL_IC,"37,332,510",0,"2,728,711","2,728,711","2,728,711","19,443,765"
UA,"41,167,335","240,896","240,896",0,"125,447","13,143,270"


## 5. Gateways

Per transfer station: journeys per day through it, LAUs reached, and the LAUs **without a direct train for which it is
the only transfer station** — the catchment that depends on it alone (PIP-15).

In [7]:
hubs_tab = []
for r in runs:
    for d in ('inbound', 'outbound'):
        nd = set(lau.loc[(lau.run == r['run']) & (lau.direction == d) & ~lau.has_direct, 'lau_gisco_id'])
        ph = per_hub[(per_hub.run == r['run']) & (per_hub.direction == d)]
        if ph.empty:
            continue
        n_h = ph.groupby('lau_gisco_id')['T'].nunique()
        only = ph[ph['lau_gisco_id'].map(n_h).eq(1) & ph['lau_gisco_id'].isin(nd)]
        t = ph.groupby('T_key').agg(journeys_per_day=('journeys_per_day', 'sum'), n_lau=('lau_gisco_id', 'nunique'))
        t['n_lau_only_hub'] = only.groupby('T_key')['lau_gisco_id'].nunique()
        t['pop_only_hub'] = only.assign(p=only['lau_gisco_id'].map(pop)).groupby('T_key')['p'].sum()
        hubs_tab.append(t.fillna({'n_lau_only_hub': 0, 'pop_only_hub': 0}).reset_index()
                        .assign(direction=d, country=r['country'], run=r['run'], **F.run_window(r)))
hubs_tab = pd.concat(hubs_tab, ignore_index=True)
hubs_tab['stop_name'] = hubs_tab['T_key'].map(geo['stop_name'])
hubs_tab['journeys_per_day'] = hubs_tab['journeys_per_day'].round(2)
hubs_tab = hubs_tab[['country', 'run', 'direction', 'T_key', 'stop_name', 'journeys_per_day', 'n_lau', 'n_lau_only_hub',
                     'pop_only_hub', 'window_start', 'window_end', 'window_days']] \
    .sort_values(['run', 'direction', 'pop_only_hub'], ascending=[True, True, False])
for r in runs:
    if r['headline']:
        display(hubs_tab[(hubs_tab.run == r['run']) & (hubs_tab.direction == 'inbound')].head(6)
                [['run', 'stop_name', 'journeys_per_day', 'n_lau', 'n_lau_only_hub', 'pop_only_hub']])
x = lau[lau.run.isin([r['run'] for r in runs if r['headline']]) & ~lau.has_direct]
print('newly reached LAUs depending on one gateway (hubs_for_90 = 1), share by run and direction:')
display(x.groupby(['run', 'direction'])['hubs_for_90'].apply(lambda s: round((s == 1).mean(), 3)).unstack())

,run,stop_name,journeys_per_day,n_lau,n_lau_only_hub,pop_only_hub
15,CZ,Praha hlavní nádraží,471.26,245,147.0,1216561.0
13,CZ,Pardubice hlavní nádraží,279.90,112,43.0,186135.0
12,CZ,Kolín,124.84,141,48.0,125056.0
6,CZ,Olomouc hlavní nádraží,272.67,116,35.0,89870.0
14,CZ,Česká Třebová,105.88,77,26.0,57991.0
8,CZ,Ostrava-Stodolní,59.00,39,7.0,30502.0


,run,stop_name,journeys_per_day,n_lau,n_lau_only_hub,pop_only_hub
68,HU,Budapest-Keleti,370.20,107,60.0,888933.0
75,HU,Hatvan,305.32,75,31.0,186813.0
77,HU,Miskolc-Tiszai,314.21,59,23.0,102880.0
76,HU,Mezőkövesd,125.16,28,1.0,290.0
69,HU,Szikszó-Vásártér,75.00,12,0.0,0.0
70,HU,Halmaj,17.07,5,0.0,0.0


,run,stop_name,journeys_per_day,n_lau,n_lau_only_hub,pop_only_hub
97,UA,Мукачево,0.73,1,0.0,0.0
98,UA,Budapest-Keleti,0.35,2,0.0,0.0
99,UA,Liptovský Mikuláš,0.57,1,0.0,0.0
100,UA,Štrba,0.57,1,0.0,0.0
101,UA,Poprad-Tatry,0.57,1,0.0,0.0
102,UA,Spišská Nová Ves,0.57,1,0.0,0.0


,run,stop_name,journeys_per_day,n_lau,n_lau_only_hub,pop_only_hub
120,PL,Bratislava hl.st.,10.00,10,8.0,806048.0
113,PL,Olomouc hlavní nádraží,2.56,9,4.0,282595.0
119,PL,Český Těšín,1.00,1,1.0,32217.0
118,PL,Poprad-Tatry,0.50,1,1.0,11012.0
114,PL,Ostrava hlavní nádraží,3.97,4,0.0,0.0
115,PL,Kolín,0.41,3,0.0,0.0


,run,stop_name,journeys_per_day,n_lau,n_lau_only_hub,pop_only_hub
139,AT,Budapest-Keleti,15.18,18,10.0,419666.0
137,AT,Bratislava hlavná stanica,77.17,16,8.0,53444.0
138,AT,Praha hlavní nádraží,1.11,7,5.0,22008.0


newly reached LAUs depending on one gateway (hubs_for_90 = 1), share by run and direction:


direction,inbound,outbound
run,,
AT,0.742,0.742
CZ,0.697,0.731
HU,0.699,0.740
PL,0.667,0.410
UA,NaN,1.000


## 6. Transfer waits and connections across midnight

The observed wait at the transfer station (STA-2), per run and direction, over the distinct journeys; and the
journeys whose second train leaves after midnight on the next date (PIP-6), with the LAUs reached **only** that way.

In [8]:
u = itins.drop_duplicates(['run', 'direction', 'lau_gisco_id', 'leg1_trip', 'leg2_trip'])
w = u.groupby(['run', 'direction'])['wait_s'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
display((w.drop(columns='count') / 60).round(1).assign(journeys=w['count'].astype(int)))
nd_only = []
for (rid, d), g in u.groupby(['run', 'direction']):
    any_l = set(g['lau_gisco_id'])
    same = set(g.loc[~g['next_day'], 'lau_gisco_id'])
    dl = set(direct.loc[(direct.run == rid) & (direct.direction == d), 'lau_gisco_id'])
    only = any_l - same - dl
    nd_only.append({'run': rid, 'direction': d, 'journeys_next_day': int(g['next_day'].sum()),
                    'lau_only_next_day': len(only), 'pop_only_next_day': pop.reindex(list(only)).sum()})
display(pd.DataFrame(nd_only))

mean   std   min   10%   25%   50%   75%   90%   max  \
run      direction                                                         
AT       inbound    61.6   5.8  22.0  56.0  56.0  64.0  65.0  66.0  66.0   
         outbound   46.6  14.0  10.0  26.0  43.0  43.0  53.0  65.0  71.0   
CZ       inbound    41.7  20.8  10.0  16.0  25.0  38.0  54.0  74.0  90.0   
         outbound   43.1  21.3  10.0  18.0  28.0  38.0  61.0  77.0  90.0   
HU       inbound    41.5  24.3  10.0  15.0  20.0  35.0  64.0  77.0  90.0   
         outbound   37.3  21.1  10.0  17.0  20.0  28.0  44.0  68.0  90.0   
PL       inbound    60.5  10.8  43.0  43.0  57.0  62.0  68.0  71.0  83.0   
         outbound   62.9  10.0  10.5  50.0  60.0  68.0  68.0  68.0  84.0   
PL:PL_IC inbound    67.2   3.7  51.0  68.0  68.0  68.0  68.0  68.0  68.0   
         outbound   65.5   4.1  10.0  60.0  60.0  68.0  68.0  68.0  74.0   
UA       inbound    40.3  18.8  10.0  21.6  25.2  38.0  55.0  63.5  72.0   
         outbound   46.5  15.6  20.0  23.6  34.5  50.0  59.0  61.2  62.0   

                    journeys  
run      direction            
AT       inbound         709  
         outbound        728  
CZ       inbound       14375  
         outbound      16453  
HU       inbound        9468  
         outbound       5771  
PL       inbound         223  
         outbound       1257  
PL:PL_IC inbound          42  
         outbound       2953  
UA       inbound          14  
         outbound         15

,run,direction,journeys_next_day,lau_only_next_day,pop_only_next_day
0,AT,inbound,295,2,53974.0
1,AT,outbound,130,9,64596.0
2,CZ,inbound,172,6,99274.0
3,CZ,outbound,6106,85,297658.0
4,HU,inbound,274,10,260847.0
5,HU,outbound,14,0,0.0
6,PL,inbound,66,2,253788.0
7,PL,outbound,929,14,628983.0
8,PL:PL_IC,inbound,0,0,0.0
9,PL:PL_IC,outbound,2883,20,1985237.0


## 7. Sensitivity

* **Transfer window** (VAL-2): the configured grid against the base window, per headline run.
* **City transfers** (PIP-17): a transfer may join **any two stations of one LAU** (Budapest-Keleti → Nyugati, Praha
  hl.n. → Masarykovo, Wien Meidling → Hbf) with a minimum of **45 min** for crossing the city. The headline keeps
  same-station transfers; the extra LAUs and population this reaches measure what that rule leaves out.

In [9]:
def sens_row(r, it, d, variant, mn, mx):
    N = len(r['days'])
    dl = set(direct.loc[(direct.run == r['run']) & (direct.direction == d), 'lau_gisco_id'])
    x = it[it.direction == d]
    s = C.summarise_one_transfer(x, 'lau_gisco_id', N) if len(x) else pd.DataFrame(columns=['lau_gisco_id', 'journeys_per_day'])
    new = x[~x['lau_gisco_id'].isin(dl)]
    _, hs = C.hub_dependence(new, 'lau_gisco_id', N) if len(new) else (None, pd.DataFrame(columns=['hubs_for_share']))
    nl = set(new['lau_gisco_id'])
    return {'country': r['country'], 'run': r['run'], 'direction': d, 'variant': variant, 'min_connect_min': mn,
            'max_connect_min': mx, 'is_base': variant == 'window' and (mn, mx) == (cfg.min_connect_min, cfg.max_connect_min),
            'n_lau_one_transfer': len(s), 'n_lau_newly_reached': len(nl), 'pop_newly_reached': pop.reindex(list(nl)).sum(),
            'journeys_per_day': round(s['journeys_per_day'].sum(), 2),
            'single_hub_share': round((hs['hubs_for_share'] == 1).mean(), 3) if len(hs) else None, **F.run_window(r)}


grid = sorted(set(cfg.sensitivity_grid) | {(cfg.min_connect_min, cfg.max_connect_min)})
sens = []
for r in runs:
    if not r['headline']:
        continue
    for mn, mx in grid:
        it = itins[itins.run == r['run']] if (mn, mx) == (cfg.min_connect_min, cfg.max_connect_min) else \
            F.run_one_transfer(cfg, r, TT[r['run']], tags, hubs, mn=mn, mx=mx)[0]
        for d in ('inbound', 'outbound'):
            sens.append(sens_row(r, it, d, 'window', mn, mx))
    city = F.run_one_transfer(cfg, r, TT[r['run']], tags, hubs, mn=45, transfer_unit='lau')[0]
    for d in ('inbound', 'outbound'):
        sens.append(sens_row(r, city, d, 'city_transfer', 45, cfg.max_connect_min))
sens = pd.DataFrame(sens)
display(sens[sens.direction == 'inbound'][['run', 'variant', 'min_connect_min', 'max_connect_min', 'is_base', 'n_lau_one_transfer',
                                            'n_lau_newly_reached', 'pop_newly_reached', 'journeys_per_day', 'single_hub_share']])
base_l = {(x.run, x.direction): set(lau.loc[(lau.run == x.run) & (lau.direction == x.direction), 'lau_gisco_id']) for x in sens.itertuples()}
print('city transfers (PIP-17): LAUs reached only with a transfer between two stations of one city, inbound:')
for r in runs:
    if r['headline']:
        city = F.run_one_transfer(cfg, r, TT[r['run']], tags, hubs, mn=45, transfer_unit='lau')[0]
        c_in = set(city.loc[city.direction == 'inbound', 'lau_gisco_id'])
        extra = c_in - base_l[(r['run'], 'inbound')] - set(direct.loc[(direct.run == r['run']) & (direct.direction == 'inbound'), 'lau_gisco_id'])
        via = city[(city.direction == 'inbound') & city['lau_gisco_id'].isin(extra)]['T'].value_counts().head(3)
        print(f"  {r['run']}: {len(extra)} LAUs, {pop.reindex(list(extra)).sum():,.0f} people; transfer cities: "
              + ', '.join(f"{lau_geo['lau_name_national'].get(t, t)} ({n})" for t, n in via.items()))

,run,variant,min_connect_min,max_connect_min,is_base,n_lau_one_transfer,n_lau_newly_reached,pop_newly_reached,journeys_per_day,single_hub_share
0,CZ,window,10,60,False,466,452,3676198.0,1726.35,0.677
2,CZ,window,10,90,True,535,521,3949238.0,2161.40,0.697
4,CZ,window,15,60,False,440,426,3600007.0,1604.44,0.709
6,CZ,window,20,60,False,432,418,3578852.0,1499.39,0.718
8,CZ,window,20,90,False,522,508,3933569.0,2033.61,0.699
10,CZ,window,20,120,False,568,554,4048101.0,2337.96,0.718
12,CZ,window,30,120,False,562,548,4043742.0,2294.70,0.681
14,CZ,city_transfer,45,90,False,595,581,4059333.0,2404.71,0.775
16,HU,window,10,60,False,149,139,1836189.0,921.43,0.770
18,HU,window,10,90,True,186,176,1978558.0,1341.82,0.699


city transfers (PIP-17): LAUs reached only with a transfer between two stations of one city, inbound:


  CZ: 80 LAUs, 205,083 people; transfer cities: Praha (1556), Ostrava (8), Olomouc (1)


  HU: 145 LAUs, 1,277,020 people; transfer cities: Budapest (11732), Bratislava (44)


  UA: 4 LAUs, 3,855,358 people; transfer cities: Budapest (4)


  PL: 0 LAUs, 0 people; transfer cities: 


  AT: 7 LAUs, 17,998 people; transfer cities: Bratislava (45)


## 8. Against the legacy run (`output/ke`)

The legacy run (`journey_efficiency_foreign_onehop.csv`) paired a ZSSK leg with the national feed at **matched
gateway stations only**, on the domestic window, with the wrong Polish feed (CF-23: PKP Intercity under the name of
PolishTrainsGTFS) and a modulo-24 h wait (PIP-6); its LAU codes are DEGURBA-2021. Compared per country: LAUs reached
(inbound, one transfer), and journeys per day for LAUs in both.

In [10]:
old = pd.read_csv('output/ke/journey_efficiency_foreign_onehop.csv', encoding='utf-8-sig')[['euc_lau_code', 'euc_lau_national', 'country', 'avg_per_day']]
new = lau[(lau.direction == 'inbound') & lau.run.isin([r['run'] for r in runs if r['headline']])]
print('LAUs per country, new (inbound, one transfer) vs legacy:')
display(pd.DataFrame({'new': new.groupby('country').size(), 'legacy': old.groupby('country').size()}).fillna(0).astype(int))
m = new.merge(old, left_on='lau_gisco_id', right_on='euc_lau_code', how='inner')
m['ratio'] = (m['journeys_per_day'] / m['avg_per_day']).round(2)
print(f'{len(m)} LAUs in both; median ratio new / legacy journeys per day by country:')
display(m.groupby('country_x')['ratio'].describe()[['count', '25%', '50%', '75%']])
print('legacy LAUs not reached now (first 15):')
display(old[~old['euc_lau_code'].isin(new['lau_gisco_id'])].head(15))

LAUs per country, new (inbound, one transfer) vs legacy:


,new,legacy
country,,
AT,31,15
CZ,535,518
HU,186,183
PL,21,11
UA,2,0


680 LAUs in both; median ratio new / legacy journeys per day by country:


,count,25%,50%,75%
country_x,,,,
AT,14.0,6.5725,19.88,26.28
CZ,485.0,1.3900,2.56,7.71
HU,170.0,5.4400,33.45,65.02
PL,11.0,1.6900,1.69,1.93


legacy LAUs not reached now (first 15):


,euc_lau_code,euc_lau_national,country,avg_per_day
41,HU_11323,Mezőkeresztes,HU,0.118132
60,HU_15307,Kápolna,HU,2.153846
63,HU_24970,Hernádszurdok,HU,11.576923
66,HU_31200,Hernádnémeti,HU,1.744505
105,CZ_575879,Uhersko,CZ,0.651099
124,CZ_575470,Poběžovice u Přelouče,CZ,2.782967
157,HU_31778,Olaszliszka,HU,5.282967
168,CZ_538680,Pyšely,CZ,3.648352
170,HU_25326,Erdőbénye,HU,5.282967
207,HU_21722,Zagyvaszántó,HU,2.019231


## 9. Outputs

In [11]:
p = os.path.join(OUT, 'one_transfer_itins.parquet')
itins.to_parquet(p, index=False)
write_meta(p, 'foreign_one_transfer_itins', itins, {'Transfer window': f'{cfg.min_connect_min}-{cfg.max_connect_min} min',
                                                     'Pairing': cfg.transfer_pairing})
print(f'[wrote] {p} ({len(itins):,} rows)')
for name, key, frame in [('one_transfer_station.csv', 'foreign_one_transfer_station', station),
                         ('one_transfer_lau.csv', 'foreign_one_transfer_lau', lau),
                         ('one_transfer_hubs.csv', 'foreign_one_transfer_hubs', hubs_tab),
                         ('one_transfer_coverage.csv', 'foreign_one_transfer_coverage', cov),
                         ('one_transfer_sensitivity.csv', 'foreign_one_transfer_sensitivity', sens),
                         ('cross_feed_duplicates.csv', 'foreign_cross_feed_duplicates', dups),
                         ('feed_quality.csv', 'foreign_feed_quality', quality)]:
    p = os.path.join(OUT, name)
    frame.to_csv(p, index=False, encoding='utf-8')
    write_meta(p, key, frame, {'Transfer window': f'{cfg.min_connect_min}-{cfg.max_connect_min} min',
                               'Pairing': cfg.transfer_pairing})
    print(f'[wrote] {p} ({len(frame):,} rows) + .meta.md')

[wrote] output\kosice\foreign\one_transfer_itins.parquet (72,346 rows)
[wrote] output\kosice\foreign\one_transfer_station.csv (2,255 rows) + .meta.md


[wrote] output\kosice\foreign\one_transfer_lau.csv (1,564 rows) + .meta.md


[wrote] output\kosice\foreign\one_transfer_hubs.csv (143 rows) + .meta.md
[wrote] output\kosice\foreign\one_transfer_coverage.csv (72 rows) + .meta.md
[wrote] output\kosice\foreign\one_transfer_sensitivity.csv (80 rows) + .meta.md
[wrote] output\kosice\foreign\cross_feed_duplicates.csv (7,010 rows) + .meta.md
[wrote] output\kosice\foreign\feed_quality.csv (12 rows) + .meta.md
